# 1) Load Data and Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_recall_curve, f1_score

import cupy as cp
import cudf
from cuml.preprocessing import StandardScaler
from cuml.ensemble import RandomForestClassifier as cuRF
from cuml.linear_model import LogisticRegression as cuLogisticRegression
from cuml.metrics import accuracy_score

import xgboost as xgb

In [2]:
df = pd.read_csv("sample_to_run_info.csv")

/tmp/ipykernel_1226/2515147133.py:1: DtypeWarning: Columns (22,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("sample_to_run_info.csv")


In [3]:
abundance = pd.read_csv('species_abundance.csv')

# 2) Exploratory Data Analysis & Cleaning

### Work with the Sample to Run Info (df)

In [4]:
# Get rid of all features that have more than 50% of their data missing
thresh = int(len(df) * 0.5)
df = df.dropna(axis=1, thresh=thresh)

In [5]:
# Sneak peek at data
df.head()

,project_id,sample_name,original_sample_description,run_id,sample_id,second_sample_id,experiment_type,nr_reads_sequenced,instrument_model,disease,phenotype,more,country,collection_date,longitude,latitude
0,PRJDB3418,NaN,APr10S00,DRR028772,DRS020620,SAMD00024580,AMPLICON,3000.0,454 GS FLX,D006262,Healthy,N,Japan,NaN,NaN,NaN
1,PRJDB3418,NaN,APr14S00,DRR028773,DRS020607,SAMD00024581,AMPLICON,3000.0,454 GS FLX,D006262,Healthy,N,Japan,NaN,NaN,NaN
2,PRJDB3418,NaN,APr15S00,DRR028774,DRS020582,SAMD00024582,AMPLICON,3000.0,454 GS FLX,D006262,Healthy,N,Japan,NaN,NaN,NaN
3,PRJDB3418,NaN,APr21S00,DRR028775,DRS020613,SAMD00024583,AMPLICON,3000.0,454 GS FLX,D006262,Healthy,N,Japan,NaN,NaN,NaN
4,PRJDB3418,NaN,APr24S00,DRR028776,DRS020600,SAMD00024584,AMPLICON,3000.0,454 GS FLX,D006262,Healthy,N,Japan,NaN,NaN,NaN


In [6]:
# Removed what is determined to be unnecessary
df = df.drop(['more', 'collection_date', 'longitude', 'latitude', 'sample_name', 'original_sample_description', 'sample_id', 'second_sample_id', 'disease'], axis=1)

In [7]:
df.head()

,project_id,run_id,experiment_type,nr_reads_sequenced,instrument_model,phenotype,country
0,PRJDB3418,DRR028772,AMPLICON,3000.0,454 GS FLX,Healthy,Japan
1,PRJDB3418,DRR028773,AMPLICON,3000.0,454 GS FLX,Healthy,Japan
2,PRJDB3418,DRR028774,AMPLICON,3000.0,454 GS FLX,Healthy,Japan
3,PRJDB3418,DRR028775,AMPLICON,3000.0,454 GS FLX,Healthy,Japan
4,PRJDB3418,DRR028776,AMPLICON,3000.0,454 GS FLX,Healthy,Japan


In [8]:
df['phenotype'].value_counts()

,count
phenotype,
Healthy,48241
Colorectal Neoplasms,5543
Crohn Disease,3516
COVID-19,2911
Parkinson Disease,2169
...,...
Colonic Polyps,5
Urolithiasis,5
Dry Eye Syndromes,4


In [9]:
# The phenotypes we will be keeping
keep_phenos = [
    "Healthy",
    "Crohn Disease",
    "Colitis, Ulcerative",
]

df = df[df["phenotype"].isin(keep_phenos)].copy()
print(df["phenotype"].value_counts())

phenotype
Healthy                48241
Crohn Disease           3516
Colitis, Ulcerative     1863
Name: count, dtype: int64


In [10]:
# A stricter filter to keep only the projects which are mainly associated with these phenotypes (Crohn's, UC, and only the healthy patients in projects that collected those two diseases).
target_phenotypes = ["Crohn Disease", "Colitis, Ulcerative"]

project_ids = (
    df[df["phenotype"].isin(target_phenotypes)]["project_id"]
    .unique()
)

df = df[df["project_id"].isin(project_ids)].copy()

print(f"Projects with at least one Crohn Disease or Ulcerative Colitis sample: {len(project_ids)}")
print(df["phenotype"].value_counts())

Projects with at least one Crohn Disease or Ulcerative Colitis sample: 46
phenotype
Crohn Disease          3516
Healthy                2099
Colitis, Ulcerative    1863
Name: count, dtype: int64


In [11]:
df['country'].value_counts()

,count
country,
United States of America,3698
China,970
Canada,666
Sweden,637
South Korea,263
Japan,196
Russia,168
Germany,167
India,101


In [ ]:
df['experiment_type'].value_counts()

# Amplicon Sequencing is a method that targets and copies specific pieces of DNA
# (amplicons) via PCR to quickly identify which microbes are present in a sample.

# Metagenomic Sequencing is a method that sequences all genetic material directly
# extracted from environmental samples to study the entire microbial community and its functions.

print(pd.crosstab(df['phenotype'], df['experiment_type'], normalize='index'))

# NOTE: this is the pre-merge picture. The species-abundance merge later (cell 23)
# drops almost all AMPLICON samples, so the modelling data is effectively
# shotgun/metagenomics only — see the experiment_type check on `merged` below.

In [ ]:
# Check whether any single phenotype is confined to (or dominated by) one sequencing
# platform, which would confound platform effects with disease signal. Only Healthy,
# Crohn, and UC remain here after the cell-11 filter; all three appear across the
# major Illumina platforms, so no phenotype is tied to a single instrument.
ct_counts = pd.crosstab(df['phenotype'], df['instrument_model'])
ct_counts

### Work with the Species Abundance (abundance)

In [14]:
abundance.head()

,id,loaded_uid,ncbi_taxon_id,taxon_rank_level,relative_abundance,accession_id
0,1,81104,-1,genus,1.95190,DRR358335
1,2,81104,544,genus,1.07457,DRR358335
2,3,81104,561,genus,0.84957,DRR358335
3,4,81104,570,genus,0.06218,DRR358335
4,5,81104,816,genus,23.29699,DRR358335


In [15]:
abundance['taxon_rank_level'].value_counts()

,count
taxon_rank_level,
genus,2780064
species,2761207


In [16]:
# Keep only species level because it is more specific than genus, allowing us to
# better pinpoint bacteria types and reasoning
abund_species = abundance[abundance["taxon_rank_level"] == "species"].copy()

# Drop rows with ncbi_taxon_id = -1 (unassigned/other)
abund_species = abund_species[abund_species["ncbi_taxon_id"] != -1].copy()

In [17]:
# We take bacteria names and make them columns, and if they exist, their values
# are relative abundances

# Pivot: run_id x ncbi_taxon_id, values = relative_abundance
abund_wide = (
    abund_species
    .pivot_table(index="accession_id",      # this matches run_id in df
                 columns="ncbi_taxon_id",
                 values="relative_abundance",
                 aggfunc="sum",
                 fill_value=0.0)
    .reset_index()
)

# Make column names nicer (e.g., taxon_544, taxon_561, ...)
abund_wide.columns = [
    "run_id" if c == "accession_id" else f"taxon_{int(c)}"
    for c in abund_wide.columns
]

### Merge the data

In [18]:
merged = df.merge(abund_wide, on="run_id", how="inner")

In [19]:
# There was 46 batches before and now there are 12
matched = df.run_id.isin(abund_wide.run_id)
missing = df[~matched]                      # ~ inverts the boolean mask
print("dropped:", len(missing))
print(missing.experiment_type.value_counts())
print(missing.phenotype.value_counts())
print("projects before:", df.project_id.nunique(), "| after:", merged.project_id.nunique())

dropped: 4640
experiment_type
AMPLICON        4444
Metagenomics     196
Name: count, dtype: int64
phenotype
Crohn Disease          2202
Healthy                1366
Colitis, Ulcerative    1072
Name: count, dtype: int64
projects before: 46 | after: 12


In [20]:
merged.head()

,project_id,run_id,experiment_type,nr_reads_sequenced,instrument_model,phenotype,country,taxon_9,taxon_17,taxon_69,...,taxon_2968459,taxon_2968968,taxon_2969304,taxon_2972461,taxon_2972466,taxon_2972769,taxon_2974552,taxon_2974597,taxon_2975484,taxon_2981769
0,PRJEB76677,ERR13295986,Metagenomics,61317544.0,Illumina HiSeq 2000,Crohn Disease,Germany,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
1,PRJEB76677,ERR13295987,Metagenomics,44948398.0,Illumina HiSeq 2000,Healthy,Germany,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.55599
2,PRJEB76677,ERR13295988,Metagenomics,7314392.0,Illumina HiSeq 2000,Crohn Disease,Germany,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000
3,PRJEB76677,ERR13295989,Metagenomics,48624378.0,Illumina HiSeq 2000,Healthy,Germany,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.45498
4,PRJEB76677,ERR13295990,Metagenomics,61732518.0,Illumina HiSeq 2000,Crohn Disease,Germany,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.01691


In [ ]:
# Settle the amplicon question on the data we actually model.
# The inner merge on run_id kept only samples with a species-abundance profile,
# which are overwhelmingly shotgun/metagenomics (nearly all AMPLICON rows were
# dropped, see cell above). So the merge already decided this for us: the modelling
# set is effectively single-technology, and species-level abundances are appropriate.
print(merged.experiment_type.value_counts())

In [21]:
# The following features, after EDA, aren't relevant
META_COLS    = ["project_id", "run_id", "phenotype", "experiment_type",
                "instrument_model", "country", "nr_reads_sequenced"]
FEATURE_COLS = [c for c in merged.columns if c.startswith("taxon_")]

# 3) Labels, Split Diagnostic, & Cross-Validation Setup

In this section we:
1. Create labels for the three phenotypes (Healthy=0, Crohn=1, UC=2)
2. **Show why a single split is untrustworthy**: with only 12 projects a grouped
   hold-out leaves just 2-3 whole cohorts in test, so every downstream metric is
   one unlucky draw.
3. Replace it with **StratifiedGroupKFold (3 folds)** — whole projects stay
   together (no cohort leakage) while labels stay balanced across folds. Every
   sample is tested exactly once, so we report **mean ± std** instead of a single
   number.

**Why 3 folds and not 5**: with 12 projects, 5 folds leaves ~2 test projects per
fold — thin enough that a fold can miss a class entirely and the per-fold std
becomes noise. 3 folds gives ~4 test projects per fold, so every fold sees all
three classes and no fold trains on a sliver. Check the geometry printout below
before trusting any score.

Feature selection, model training, XGBoost early stopping, and threshold tuning
all happen **inside each fold** on training/validation data only — the held-out
fold is touched once, for scoring.

In [ ]:
keep_phenos = ["Healthy", "Crohn Disease", "Colitis, Ulcerative"]
# reset_index so positional (iloc) indices from the CV splitter line up with the frame
df3 = merged[merged["phenotype"].isin(keep_phenos)].copy().reset_index(drop=True)

# We label them to 0, 1, and 2 for multiclass labelling
label_map = {
    "Healthy": 0,
    "Crohn Disease": 1,
    "Colitis, Ulcerative": 2,
}
df3["label"] = df3["phenotype"].map(label_map)

taxon_cols = [c for c in df3.columns if c.startswith("taxon_")]
print("samples:", len(df3), "| taxa:", len(taxon_cols), "| projects:", df3.project_id.nunique())

In [ ]:
# Item 2 — confirm how thin a single grouped split really is, BEFORE trusting it.
# With ~12 projects, one 80/20 grouped hold-out puts only 2-3 whole cohorts in test.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df3, df3["label"], groups=df3["project_id"]))
df_train, df_test = df3.iloc[train_idx], df3.iloc[test_idx]

print(df_train.project_id.nunique(), "train |", df_test.project_id.nunique(), "test")
print(df_test.label.value_counts())   # raw counts, not proportions: how few samples decide each metric

In [ ]:
# Cross-validation setup — this replaces the single split above.
# StratifiedGroupKFold keeps whole projects together (no cohort leakage) while
# balancing labels across folds, so every sample is tested exactly once.
from sklearn.model_selection import StratifiedGroupKFold

groups = df3["project_id"].to_numpy()
y_all  = df3["label"].to_numpy()


def fold_problems(splitter, frame, y, grp, min_train_frac=0.4, min_class_count=10):
    """Return a list of reasons this fold geometry is unusable ([] means it's fine).

    StratifiedGroupKFold balances LABELS but not SAMPLE COUNTS, and when a class
    lives in only a few projects it cannot keep that class in every fold at all.
    Two distinct failure modes, both seen in this dataset:

      * class missing from a fold's TEST set  -> that fold's AUC is undefined
      * class missing from a fold's TRAIN set -> the model cannot be fit at all
        (cuML raises "loss='sigmoid' requires n_classes == 2 (got 1)")

    A fold that trains on a sliver is a third, quieter failure: at 5 folds one fold
    trained on 438 samples and produced predictions for 2400.
    """
    n = len(frame)
    classes = set(np.unique(y).tolist())
    problems = []

    for f, (tr, te) in enumerate(splitter.split(frame, y, grp)):
        missing_tr = sorted(classes - set(np.unique(y[tr]).tolist()))
        if missing_tr:
            problems.append(f"fold {f}: TRAIN missing class(es) {missing_tr} — cannot fit")

        counts = pd.Series(y[te]).value_counts().to_dict()
        thin = [int(c) for c in sorted(classes) if counts.get(c, 0) < min_class_count]
        if thin:
            problems.append(f"fold {f}: TEST has <{min_class_count} samples of class(es) {thin}")

        if len(tr) < min_train_frac * n:
            problems.append(f"fold {f}: trains on only {len(tr)}/{n} ({len(tr)/n:.0%})")

    return problems


def show_fold_geometry(splitter, frame, y, grp, title=""):
    """Print each fold's shape and any problems found."""
    if title:
        print(f"=== {title} ===")
    for f, (tr, te) in enumerate(splitter.split(frame, y, grp)):
        counts = pd.Series(y[te]).value_counts().sort_index().to_dict()
        print(f"fold {f}: {len(tr):4d} train / {len(te):4d} test | "
              f"test projects={frame.iloc[te]['project_id'].nunique()} | test counts={counts}")
    problems = fold_problems(splitter, frame, y, grp)
    if problems:
        print("!! DEGENERATE FOLDS — scores below are not interpretable:")
        for p in problems:
            print("   -", p)
    else:
        print("Fold geometry OK: all classes present in every train and test fold.")
    return problems


def choose_splitter(frame, y, grp, n_splits_options=(3, 2), seeds=(42, 0, 1, 7, 13, 2024)):
    """Pick the first (n_splits, seed) whose folds are all usable.

    With 12 projects and a class concentrated in a few of them, not every setting
    yields a workable split — so search instead of hard-coding one and hoping.
    """
    best, best_problems = None, None
    for ns in n_splits_options:
        for s in seeds:
            sp = StratifiedGroupKFold(n_splits=ns, shuffle=True, random_state=s)
            problems = fold_problems(sp, frame, y, grp)
            if not problems:
                print(f"selected n_splits={ns}, random_state={s} (clean geometry)")
                return sp
            if best is None or len(problems) < len(best_problems):
                best, best_problems = (sp, ns, s), problems
    sp, ns, s = best
    print(f"WARNING: no fully clean geometry found. Using the least-bad option "
          f"(n_splits={ns}, random_state={s}) with {len(best_problems)} problem(s).")
    return sp


# Fix 1 — fold geometry. 5 folds over ~12 projects produced folds with 5 test samples,
# folds missing a class entirely, and one fold training on 15% of the data. Fewer folds
# means more projects per fold; we search for a setting that is actually usable.
sgkf = choose_splitter(df3, y_all, groups)
N_SPLITS = sgkf.get_n_splits()
_ = show_fold_geometry(sgkf, df3, y_all, groups, "multiclass fold geometry")


def grouped_val_split(train_idx, y, grp, val_frac=0.25, seed=42):
    """Carve a group-disjoint validation set out of a fold's training indices.

    Used for XGBoost early stopping and threshold tuning so the held-out test
    fold is never consulted during training/tuning. Falls back to a stratified
    non-grouped split if the grouped one would leave a class out of validation.
    """
    gss_inner = GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    inner, val = next(gss_inner.split(train_idx, y[train_idx], grp[train_idx]))
    inner_idx, val_idx = train_idx[inner], train_idx[val]

    # Threshold tuning and early stopping both need both classes on each side.
    n_classes = len(np.unique(y[train_idx]))
    if len(np.unique(y[inner_idx])) < n_classes or len(np.unique(y[val_idx])) < n_classes:
        rng = np.random.default_rng(seed)
        perm = rng.permutation(len(train_idx))
        cut = int(len(train_idx) * (1 - val_frac))
        inner_idx, val_idx = train_idx[perm[:cut]], train_idx[perm[cut:]]
        print("   note: grouped validation split left a class empty; "
              "fell back to a random (non-grouped) validation split for this fold.")
    return inner_idx, val_idx

In [ ]:
def require_scored(res_dict, name="this CV"):
    """Return the `scored` mask, failing with a readable explanation if it is empty.

    Every fold being skipped is a real possibility here, not a coding slip: it means
    the minority class is confined to so few projects that grouped CV cannot leave
    any of it in a training fold. Better to say that plainly than to let the report
    cells die on empty arrays.
    """
    m = res_dict["scored"]
    if not m.any():
        raise RuntimeError(
            f"No samples in {name} were scored — every fold had a single-class "
            "training set. The minority class is confined to too few projects for "
            "grouped cross-validation to work at any fold count. See the "
            "project x label crosstab in Section 8; this is a data limitation, "
            "not a bug to fix in code."
        )
    return m

# 4) Per-Fold L1 Feature Selection (LASSO)

An L1-regularized logistic regression selects a sparse taxa set. Crucially this is
now defined as a helper that is **refit inside every CV fold on that fold's training
cohorts only** — the union of non-zero coefficients across classes becomes the
feature set for that fold, so the held-out projects never influence which features
are chosen.

In [ ]:
def select_features(X_tr, y_tr, C=0.1):
    """Fit L1 logistic regression on training rows; return taxa with any non-zero coef.

    X_tr : 2D numpy array of abundances for the training rows, columns in
           `taxon_cols` order. Already feature-transformed by the caller when a
           transform is in use — LASSO is sensitive to the transform even though
           the tree models are not, so it must see the same representation.
    y_tr : integer labels for those rows. Must contain at least two classes;
           cuML raises "loss='sigmoid' requires n_classes == 2 (got 1)" otherwise,
           which is why the CV loops skip single-class training folds.
    """
    scaler = StandardScaler(with_mean=False)
    X_tr_scaled = scaler.fit_transform(cudf.DataFrame(X_tr, columns=taxon_cols))

    clf = cuLogisticRegression(
        penalty="l1",
        C=C,                   # controls sparsity
        l1_ratio=None,         # pure L1
        fit_intercept=True,
        class_weight="balanced",
        max_iter=5000,
        solver="qn",
        verbose=0,
    )
    clf.fit(X_tr_scaled, cudf.Series(y_tr))

    coefs = clf.coef_.to_pandas().to_numpy()   # (n_classes, n_features)
    nz_any = (coefs != 0).any(axis=0)          # keep taxa non-zero for any class
    return list(np.array(taxon_cols)[nz_any])

# 5) Cross-Validated Multiclass Evaluation (Random Forest + XGBoost)

One pass over the StratifiedGroupKFold folds. In each fold we:
1. apply the feature transform, then select features with LASSO on the **training
   cohorts only** (`select_features`),
2. carve a **grouped validation set** out of the training rows (`grouped_val_split`),
3. train Random Forest on the fold's training data,
4. train XGBoost with **early stopping on the validation set** (never the test fold),
5. predict the held-out fold once and store the predictions.

Because the folds are disjoint, stacking the per-fold predictions gives one
**out-of-fold (OOF)** prediction per sample. We report per-fold accuracy/AUC as
**mean ± std**, compare feature transforms (raw / log1p / CLR), and test class
weighting as a lever on UC recall.

In [ ]:
import inspect

_RF_KW = dict(n_estimators=300, max_depth=20, max_features=1.0,
              n_bins=16, bootstrap=True, random_state=42)

# Not every cuML build exposes class_weight on the RF; detect instead of assuming.
_RF_SUPPORTS_CW = "class_weight" in inspect.signature(cuRF.__init__).parameters
if not _RF_SUPPORTS_CW:
    print("note: this cuML build's RandomForestClassifier has no class_weight; "
          "RF will run unweighted (XGBoost still uses sample weights).")


def make_rf(balance=False):
    if balance and _RF_SUPPORTS_CW:
        return cuRF(class_weight="balanced", **_RF_KW)
    return cuRF(**_RF_KW)


def balanced_weights(y):
    """Per-sample weights inversely proportional to class frequency.

    XGBoost's multi:softprob has no class_weight, so we pass weights on the DMatrix.
    This is the multiclass equivalent of class_weight='balanced'.
    """
    classes, counts = np.unique(y, return_counts=True)
    freq = dict(zip(classes, counts))
    return np.array([len(y) / (len(classes) * freq[v]) for v in y], dtype=float)


def run_multiclass_cv(transform=None, label="raw", balance=False):
    """Grouped CV (N_SPLITS folds) for the 3-class problem.

    transform : row-wise feature transform (np.log1p, clr, ...), applied BEFORE
        feature selection so the whole pipeline sees one representation. These
        transforms use no cross-sample statistics, so transforming the full matrix
        up front is NOT leakage (unlike, say, a StandardScaler fit on all rows).
    balance : weight classes inversely to frequency, to lift minority-class recall.

    Returns out-of-fold predictions plus a `scored` mask marking which samples
    actually received a prediction (a fold with a single-class training set is
    skipped, so its samples are left unscored rather than silently counted wrong).
    """
    X_full = df3[taxon_cols].to_numpy()
    if transform is not None:
        X_full = transform(X_full)
    pos = {c: i for i, c in enumerate(taxon_cols)}

    n = len(df3)
    oof_pred_rf   = np.full(n, -1)
    oof_pred_xgb  = np.full(n, -1)
    oof_proba_xgb = np.zeros((n, 3))
    scored        = np.zeros(n, dtype=bool)
    rf_acc, xgb_acc, xgb_auc = [], [], []

    for fold_i, (tr, te) in enumerate(sgkf.split(df3, y_all, groups)):
        # A classifier needs at least two classes in training; skip rather than crash.
        if len(np.unique(y_all[tr])) < 2:
            print(f"   [{label}] skipping fold {fold_i}: training set has a single class; "
                  f"its {len(te)} test samples are left unscored.")
            continue

        # (1) feature selection on the TRANSFORMED training rows only
        feats = select_features(X_full[tr], y_all[tr])
        fp = [pos[f] for f in feats]

        # (2) grouped validation carve-out for XGBoost early stopping
        inner, val = grouped_val_split(tr, y_all, groups)

        X_tr    = X_full[np.ix_(tr, fp)]
        X_inner = X_full[np.ix_(inner, fp)]
        X_val   = X_full[np.ix_(val, fp)]
        X_te    = X_full[np.ix_(te, fp)]
        y_te = y_all[te]

        # (3) Random Forest — trains on the full fold-train (no early stopping needed)
        rf = make_rf(balance)
        rf.fit(cudf.DataFrame(X_tr, columns=feats), cudf.Series(y_all[tr]))
        pred_rf = rf.predict(cudf.DataFrame(X_te, columns=feats)).to_numpy().astype(int)
        oof_pred_rf[te] = pred_rf
        rf_acc.append((pred_rf == y_te).mean())

        # (4) XGBoost — early stop on the grouped validation set, NOT on test
        dtrain = xgb.DMatrix(X_inner, label=y_all[inner],
                             weight=balanced_weights(y_all[inner]) if balance else None)
        dval   = xgb.DMatrix(X_val, label=y_all[val],
                             weight=balanced_weights(y_all[val]) if balance else None)
        dtest  = xgb.DMatrix(X_te, label=y_te)
        params = {
            "objective": "multi:softprob", "num_class": 3,
            "tree_method": "hist", "max_depth": 6, "eta": 0.1,
            "subsample": 0.8, "colsample_bytree": 0.8,
            "eval_metric": "mlogloss", "device": "cuda",
        }
        bst = xgb.train(params, dtrain, num_boost_round=300,
                        evals=[(dval, "val")], early_stopping_rounds=30,
                        verbose_eval=False)

        # (5) predict the held-out fold once
        proba = bst.predict(dtest)
        oof_proba_xgb[te] = proba
        pred_xgb = proba.argmax(axis=1)
        oof_pred_xgb[te] = pred_xgb
        scored[te] = True
        xgb_acc.append((pred_xgb == y_te).mean())
        if len(np.unique(y_te)) == 3:
            xgb_auc.append(roc_auc_score(np.eye(3)[y_te], proba, multi_class="ovr"))
        else:
            xgb_auc.append(np.nan)   # AUC undefined when a class is absent from test

    def ms(a):
        return f"{np.nanmean(a):.3f} ± {np.nanstd(a):.3f}" if len(a) else "n/a"
    print(f"[{label}] RF  accuracy {ms(rf_acc)}")
    print(f"[{label}] XGB accuracy {ms(xgb_acc)} | macro ROC-AUC {ms(xgb_auc)}")
    if not scored.all():
        print(f"[{label}] WARNING: {(~scored).sum()}/{n} samples unscored (skipped folds).")

    return {
        "oof_pred_rf": oof_pred_rf, "oof_pred_xgb": oof_pred_xgb,
        "oof_proba_xgb": oof_proba_xgb, "scored": scored,
        "rf_acc": rf_acc, "xgb_acc": xgb_acc, "xgb_auc": xgb_auc,
    }

In [ ]:
# Item 6 / Fix 2 — feature engineering, wired up properly this time.
#
# WHY THE OLD log1p COMPARISON WAS A NO-OP (two compounding reasons):
#   1. Tree ensembles split on thresholds (x < t), so they are INVARIANT to any
#      monotonic per-feature transform. log1p is monotonic => RF/XGBoost produce
#      identical trees on raw and log1p data. This is maths, not a bug — and it is
#      why the two runs printed byte-identical scores (0.651 ± 0.215 both times).
#   2. The only step that *is* sensitive to it — the LASSO selection — was being fed
#      RAW values regardless of `transform`. That was a genuine wiring gap, now fixed:
#      run_multiclass_cv applies the transform BEFORE select_features.
# So log1p can now move results, but only via which taxa LASSO picks.
#
# CLR (centered log-ratio) is the principled transform for compositional data such as
# relative abundances: it divides each taxon by that sample's geometric mean. Because
# it mixes information ACROSS taxa within a sample, it is NOT a per-feature monotonic
# map, so it changes the trees themselves as well as the selection.
def clr(X, pseudocount=1e-6):
    """Centered log-ratio transform for compositional (relative abundance) rows."""
    Xp = np.asarray(X, dtype=float) + pseudocount
    log_Xp = np.log(Xp)
    return log_Xp - log_Xp.mean(axis=1, keepdims=True)


res_raw   = run_multiclass_cv(transform=None,     label="raw")
res_log1p = run_multiclass_cv(transform=np.log1p, label="log1p")
res_clr   = run_multiclass_cv(transform=clr,      label="clr")

# Choose what the rest of the notebook reports. Keep these two lines in sync —
# BEST_TRANSFORM is reused for the feature-importance refit in Section 6 so the
# importances describe the same representation the reported scores came from.
res            = res_raw
BEST_TRANSFORM = None      # set to np.log1p or clr if one clearly wins above

In [ ]:
# Out-of-fold report: every scored sample was predicted exactly once across the folds,
# so this single report summarises them without any test leakage.
# `scored` excludes samples from any fold that had to be skipped — including them
# would count an absent prediction as a wrong one.
target_names = ["Healthy", "Crohn", "UC"]
m = require_scored(res, "the multiclass CV")
if not m.all():
    print(f"NOTE: reporting on {m.sum()}/{len(m)} samples; the rest were in skipped folds.\n")

print("Random Forest — out-of-fold")
print(classification_report(y_all[m], res["oof_pred_rf"][m], target_names=target_names))

print("XGBoost — out-of-fold")
print(classification_report(y_all[m], res["oof_pred_xgb"][m], target_names=target_names))
if len(np.unique(y_all[m])) == 3:
    print("XGBoost macro ROC-AUC (OOF):",
          roc_auc_score(np.eye(3)[y_all[m]], res["oof_proba_xgb"][m], multi_class="ovr"))
else:
    print("Macro ROC-AUC skipped: not all three classes are present among scored samples.")

In [ ]:
# Fix 3 — UC recall. UC is the minority class and is clinically the hardest to
# separate from Crohn, so it carries the weakest recall above. Try class weighting:
# RF via class_weight='balanced' (where this cuML build supports it), XGBoost via
# per-sample weights, since multi:softprob has no class_weight parameter.
from sklearn.metrics import recall_score

res_bal = run_multiclass_cv(transform=None, label="raw+balanced", balance=True)


def per_class_recall(r, key):
    mask = r["scored"]
    return recall_score(y_all[mask], r[key][mask],
                        labels=[0, 1, 2], average=None, zero_division=0)


print("\nrecall per class [Healthy, Crohn, UC] — UC is the number to watch:")
for name, r in [("unweighted", res_raw), ("balanced", res_bal)]:
    print(f"  {name:>10} | RF  {per_class_recall(r, 'oof_pred_rf').round(3)}"
          f" | XGB {per_class_recall(r, 'oof_pred_xgb').round(3)}")

# Expect a modest lift at best. Class weighting redistributes the decision boundary;
# it cannot create signal that isn't in the data. If UC recall stays low, that is the
# finding — see the limitations note at the end of the notebook.

# 6) Feature Importances (interpretation only)

The CV above gives the honest performance estimate. To ask *which taxa matter*, we
refit the pipeline once on **all** samples — feature selection, Random Forest, and
XGBoost — and read off importances. These are for biological interpretation only;
they are **not** a performance number, because no data was held out here.

In [ ]:
# Refit on all data purely to rank features (do NOT read accuracy into this).
# Uses BEST_TRANSFORM so the importances describe the same representation as the
# reported CV scores — ranking CLR features while reporting raw scores would be
# comparing two different models.
X_taxa = df3[taxon_cols].to_numpy()
if BEST_TRANSFORM is not None:
    X_taxa = BEST_TRANSFORM(X_taxa)

all_feats = select_features(X_taxa, y_all)
print("Features selected on full data:", len(all_feats))

pos_all = {c: i for i, c in enumerate(taxon_cols)}
X_all_sel = X_taxa[:, [pos_all[f] for f in all_feats]]

rf_full = cuRF(**_RF_KW)
rf_full.fit(cudf.DataFrame(X_all_sel, columns=all_feats), cudf.Series(y_all))

dall = xgb.DMatrix(X_all_sel, label=y_all, feature_names=all_feats)
params_full = {
    "objective": "multi:softprob", "num_class": 3, "tree_method": "hist",
    "max_depth": 6, "eta": 0.1, "subsample": 0.8, "colsample_bytree": 0.8,
    "eval_metric": "mlogloss", "device": "cuda",
}
bst_full = xgb.train(params_full, dall, num_boost_round=200)

In [ ]:
# Random Forest importances (mean impurity decrease)
importances = np.asarray(rf_full.feature_importances_)
idx = importances.argsort()[::-1][:30]
rf_imp_df = pd.DataFrame({
    "taxon": np.array(all_feats)[idx],
    "rf_importance": importances[idx],
})
print(rf_imp_df)

In [ ]:
# XGBoost importances (gain = average improvement in loss from splits on that taxon)
gain = bst_full.get_score(importance_type="gain")   # keys are already taxon names
xgb_imp_df = (
    pd.DataFrame(gain.items(), columns=["taxon", "gain"])
      .sort_values("gain", ascending=False)
      .head(30)
      .reset_index(drop=True)
)
print(xgb_imp_df)

# 7) Out-of-Fold Confusion Matrices

Confusion matrices built from the **out-of-fold** predictions — i.e. pooled over the
folds, so every sample is counted once and none was seen during its own training.

In [46]:
class_names = ["Healthy", "Crohn", "UC"]

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names,
                yticklabels=class_names,
                cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    print(title)
    print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Random Forest — out-of-fold confusion matrix (scored samples only)
m = res["scored"]
plot_cm(y_all[m], res["oof_pred_rf"][m], "Random Forest (out-of-fold)")

In [ ]:
# XGBoost — out-of-fold confusion matrix (scored samples only)
m = res["scored"]
plot_cm(y_all[m], res["oof_pred_xgb"][m], "XGBoost (out-of-fold)")

# 8) Binary Classification: Crohn Disease vs Ulcerative Colitis

Restricted to the IBD samples (Healthy excluded), 0 = Crohn, 1 = UC. Same discipline
as the multiclass section: its own **StratifiedGroupKFold** over the IBD projects, and
inside each fold we
- select features on training cohorts only,
- carve a **grouped validation set**,
- **early-stop XGBoost on validation** (fixes the old leak of early-stopping on test),
- **tune the decision threshold on validation** (fixes the old leak of tuning on test),
- predict the held-out fold once.

The threshold matters here because the classes are imbalanced (far more Crohn than
UC), so the default 0.5 cutoff is rarely the best F1 operating point.

In [ ]:
# Crohn vs UC only — Healthy excluded. Build the IBD subset and its own grouped CV.
ibd_mask = df3["phenotype"].isin(["Crohn Disease", "Colitis, Ulcerative"]).to_numpy()
df_ibd = df3[ibd_mask].reset_index(drop=True)

groups_ibd = df_ibd["project_id"].to_numpy()
# 0 = Crohn, 1 = UC  (matches target_names=["Crohn", "UC"])
y_ibd = (df_ibd["phenotype"] == "Colitis, Ulcerative").astype(int).to_numpy()

print("IBD samples:", len(df_ibd), "| projects:", df_ibd.project_id.nunique())
print("label counts (0=Crohn, 1=UC):", pd.Series(y_ibd).value_counts().sort_index().to_dict())

# ROOT-CAUSE DIAGNOSTIC: how is UC spread across projects?
# Grouped CV can only keep UC in every fold if UC appears in enough projects. If UC
# is concentrated in a handful of cohorts, some fold's TRAINING set ends up pure
# Crohn and the model cannot be fit at all — cuML fails with
#   "loss='sigmoid' requires n_classes == 2 (got 1)".
# That crash is not a coding bug; it is this table being too sparse.
ibd_ct = pd.crosstab(df_ibd["project_id"], y_ibd)
ibd_ct.columns = ["Crohn", "UC"]
print("\nsamples per project:")
print(ibd_ct)
n_uc_projects = int((ibd_ct["UC"] > 0).sum())
print(f"\nprojects containing UC: {n_uc_projects} of {len(ibd_ct)}")
if n_uc_projects < 4:
    print("=> UC is concentrated in very few cohorts. Grouped CV is severely limited "
          "here, and this is a data constraint, not something modelling can fix.")

In [ ]:
# Pick a usable geometry for the IBD problem specifically. It needs its own search:
# UC lives in fewer projects than the 3-class labels do, so a setting that works for
# the multiclass CV can still leave a binary fold with no UC to train on.
sgkf_ibd = choose_splitter(df_ibd, y_ibd, groups_ibd)
_ = show_fold_geometry(sgkf_ibd, df_ibd, y_ibd, groups_ibd, "binary (Crohn vs UC) fold geometry")

In [ ]:
def run_binary_cv(transform=None, balance=False):
    """Grouped CV for Crohn (0) vs UC (1).

    Per fold: LASSO on train only, grouped validation carve-out, XGBoost early-stopped
    on validation, decision threshold tuned on validation, then the test fold scored
    once. transform is applied BEFORE feature selection (same Fix 2 wiring as the
    multiclass loop); it is row-wise, so transforming up front is not leakage.

    Folds whose TRAINING set contains only one class are skipped — a binary
    classifier cannot be fit on them (this is what raised
    "loss='sigmoid' requires n_classes == 2 (got 1)"). The returned `scored` mask
    records which samples actually got a prediction.
    """
    X_full = df_ibd[taxon_cols].to_numpy()
    if transform is not None:
        X_full = transform(X_full)
    pos = {c: i for i, c in enumerate(taxon_cols)}

    n = len(df_ibd)
    oof_proba   = np.zeros(n)          # XGB probability of UC per sample
    oof_thr     = np.full(n, 0.5)      # each fold's validation-tuned threshold
    oof_pred_rf = np.full(n, -1)       # RF prediction (default 0.5)
    scored      = np.zeros(n, dtype=bool)
    fold_auc, fold_thr = [], []

    for fold_i, (tr, te) in enumerate(sgkf_ibd.split(df_ibd, y_ibd, groups_ibd)):
        if len(np.unique(y_ibd[tr])) < 2:
            print(f"   skipping fold {fold_i}: training set is single-class "
                  f"({len(tr)} samples, all class {int(y_ibd[tr][0])}); "
                  f"its {len(te)} test samples are left unscored.")
            continue

        feats = select_features(X_full[tr], y_ibd[tr])
        fp = [pos[f] for f in feats]

        inner, val = grouped_val_split(tr, y_ibd, groups_ibd)

        X_inner = X_full[np.ix_(inner, fp)]
        X_val   = X_full[np.ix_(val, fp)]
        X_te    = X_full[np.ix_(te, fp)]
        X_tr    = X_full[np.ix_(tr, fp)]

        # XGBoost — early stop on validation, not test (leak #2 fixed)
        dtrain = xgb.DMatrix(X_inner, label=y_ibd[inner],
                             weight=balanced_weights(y_ibd[inner]) if balance else None)
        dval   = xgb.DMatrix(X_val, label=y_ibd[val],
                             weight=balanced_weights(y_ibd[val]) if balance else None)
        dtest  = xgb.DMatrix(X_te, label=y_ibd[te])
        params_binary = {
            "objective": "binary:logistic", "tree_method": "hist", "device": "cuda",
            "max_depth": 6, "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.8,
            "eval_metric": "auc",
        }
        bst = xgb.train(params_binary, dtrain, num_boost_round=500,
                        evals=[(dval, "val")], early_stopping_rounds=30,
                        verbose_eval=False)

        # Tune the decision threshold on validation, not test (leak #3 fixed).
        # A single-class validation fold makes the PR curve meaningless — keep 0.5.
        val_proba = bst.predict(dval)
        if len(np.unique(y_ibd[val])) < 2:
            best_thr = 0.5
            print(f"   fold {fold_i}: validation set is single-class; "
                  f"keeping the default 0.5 threshold instead of tuning.")
        else:
            prec, rec, thr = precision_recall_curve(y_ibd[val], val_proba)
            f1 = 2 * prec * rec / (prec + rec + 1e-8)
            # thr has len n-1 vs prec/rec len n; drop the last point (no threshold)
            best_thr = float(thr[np.argmax(f1[:-1])]) if len(thr) else 0.5
        fold_thr.append(best_thr)

        # Score the held-out fold once
        te_proba = bst.predict(dtest)
        oof_proba[te] = te_proba
        oof_thr[te]   = best_thr
        scored[te]    = True
        if len(np.unique(y_ibd[te])) == 2:
            fold_auc.append(roc_auc_score(y_ibd[te], te_proba))
        else:
            fold_auc.append(np.nan)   # AUC undefined on a single-class test fold

        # Random Forest baseline (default 0.5 threshold)
        rf = make_rf(balance)
        rf.fit(cudf.DataFrame(X_tr, columns=feats), cudf.Series(y_ibd[tr]))
        oof_pred_rf[te] = rf.predict(cudf.DataFrame(X_te, columns=feats)).to_numpy().astype(int)

    if fold_auc:
        print(f"XGB per-fold ROC-AUC: {np.nanmean(fold_auc):.3f} ± {np.nanstd(fold_auc):.3f}"
              f"  (over {int(np.sum(~np.isnan(fold_auc)))} usable fold(s))")
    print(f"validation-tuned thresholds per fold: {[round(t, 3) for t in fold_thr]}")
    if not scored.all():
        print(f"WARNING: {(~scored).sum()}/{n} samples unscored (skipped folds).")

    return {"oof_proba": oof_proba, "oof_thr": oof_thr, "oof_pred_rf": oof_pred_rf,
            "scored": scored, "fold_auc": fold_auc, "fold_thr": fold_thr}


res_ibd = run_binary_cv()

In [ ]:
def plot_cm_binary(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=["Crohn", "UC"],
                yticklabels=["Crohn", "UC"],
                cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# XGBoost out-of-fold evaluation at the default 0.5 threshold (scored samples only)
mb = require_scored(res_ibd, "the binary Crohn-vs-UC CV")
if not mb.all():
    print(f"NOTE: reporting on {mb.sum()}/{len(mb)} IBD samples; "
          f"the rest were in skipped folds.\n")

y_pred_ibd = (res_ibd["oof_proba"][mb] > 0.5).astype(int)
print("XGBoost — Crohn vs UC (out-of-fold, threshold 0.5)")
print(classification_report(y_ibd[mb], y_pred_ibd, target_names=["Crohn", "UC"]))
if len(np.unique(y_ibd[mb])) == 2:
    print(f"OOF ROC-AUC: {roc_auc_score(y_ibd[mb], res_ibd['oof_proba'][mb]):.4f}")
else:
    print("OOF ROC-AUC skipped: only one class present among scored samples.")
plot_cm_binary(y_ibd[mb], y_pred_ibd, "XGBoost — Crohn vs UC (OOF, 0.5)")

In [ ]:
# Random Forest out-of-fold evaluation (baseline, default threshold)
mb = require_scored(res_ibd, "the binary Crohn-vs-UC CV")
print("Random Forest — Crohn vs UC (out-of-fold)")
print(classification_report(y_ibd[mb], res_ibd["oof_pred_rf"][mb], target_names=["Crohn", "UC"]))
plot_cm_binary(y_ibd[mb], res_ibd["oof_pred_rf"][mb], "Random Forest — Crohn vs UC (OOF)")

In [ ]:
# Threshold tuning, reported honestly: each fold's threshold was chosen on that fold's
# VALIDATION set, then applied to its own held-out test samples. Here we pool those
# out-of-fold predictions and report once — no cutoff was ever fit on test data.
mb = require_scored(res_ibd, "the binary Crohn-vs-UC CV")
y_pred_default = (res_ibd["oof_proba"][mb] > 0.5).astype(int)
y_pred_tuned   = (res_ibd["oof_proba"][mb] > res_ibd["oof_thr"][mb]).astype(int)

print(f"Default threshold (0.5):        F1 = {f1_score(y_ibd[mb], y_pred_default):.4f}")
print(f"Validation-tuned per-fold:      F1 = {f1_score(y_ibd[mb], y_pred_tuned):.4f}")
print("\nTuned-threshold out-of-fold results:")
print(classification_report(y_ibd[mb], y_pred_tuned, target_names=["Crohn", "UC"]))
plot_cm_binary(y_ibd[mb], y_pred_tuned, "XGBoost tuned (val) — Crohn vs UC (OOF)")

# Read this as a TRADE, not an improvement: lowering the cutoff buys UC recall by
# giving up Crohn recall. Which side to favour is a clinical decision, not a
# modelling one — the ROC-AUC above is unchanged by the threshold.

# 9) Findings & Limitations

**Headline result: report the binary Crohn vs UC model as the main IBD finding.**
It is the model with enough signal and enough support per class to be worth quoting,
and it is evaluated cleanly — grouped folds, early stopping and threshold tuning on
validation only, every sample scored out-of-fold exactly once.

**Three-class UC detection is at or near the limit of this dataset.** UC recall stays
low in the 3-class setting and class weighting moves it only marginally. That is a
finding, not a bug, and it has concrete causes:

- **UC is the smallest class** and is competing against Crohn, which it genuinely
  resembles at the species-abundance level — the two are hard to separate clinically
  as well as computationally.
- **Only ~12 projects survive the abundance merge**, so cohort (batch) effects are
  large relative to the disease signal, and grouped CV correctly refuses to reward a
  model for memorising them.
- **Class weighting redistributes a decision boundary; it cannot manufacture signal.**
  It trades Healthy/Crohn recall for UC recall rather than adding information.

**What would actually fix it — more data — is not available here.** Specifically: more
independent cohorts (not more samples from the same 12 projects), since the binding
constraint is the number of projects, not the number of rows.

**How to read the numbers in this notebook.** Quote the out-of-fold results with their
per-fold **mean ± std**, not a single split's accuracy. The std is part of the result:
with this few cohorts it is wide, and a difference smaller than the spread between
folds is not a real difference. Treat the feature importances in Section 6 as
hypothesis-generating only — they come from an all-data refit with no hold-out.